# 테이블 불러오기

In [0]:
agro_df = spark.table("silver.agrofood.agrofood_normalized2").dropDuplicates()
display(agro_df)

# -----------------------------------------------------------------------------------------------

In [0]:
display(
    agro_df.select("카테고리").distinct().orderBy("카테고리")
)

In [0]:
from pyspark.sql.functions import when

agro_df = agro_df.withColumn(
    "카테고리",
    when(agro_df["카테고리"] == "식량작물", "쌀/잡곡")
    .otherwise(agro_df["카테고리"])
)
display(agro_df)

In [0]:
from pyspark.sql.functions import when

agro_df = agro_df.withColumn(
    "카테고리",
    when(agro_df["재료명"] == "감자", "채소류").otherwise(agro_df["카테고리"])
)
display(agro_df)

In [0]:
display(
    agro_df.filter(agro_df["카테고리"] == "축산물")
           .select("재료명", "세부속성")
           .distinct()
           .orderBy("재료명", "세부속성")
)

In [0]:
display(
    agro_df.select("단위_문자").distinct().orderBy("단위_문자")
)

In [0]:
agro_df = agro_df.withColumn(
    "단위_문자",
    when(agro_df["단위_문자"] == "kg(그물망 3포기)", "kg")
    .when(agro_df["단위_문자"] == "10kg", "kg")
    .when(agro_df["단위_문자"] == "리터", "L")
    .otherwise(agro_df["단위_문자"])
)
display(
    agro_df
)

In [0]:
agro_df = agro_df.withColumn(
    "단위",
    when(agro_df["단위"] == "10kg(그물망 3포기)", "10kg")
    .when(agro_df["단위"].like ("리터"), ("L"))
    .otherwise(agro_df["단위"])
)

In [0]:
display(
    agro_df.select("단위_문자").distinct().orderBy("단위_문자")
)

In [0]:
from pyspark.sql.functions import when

agro_df = agro_df.withColumn(
    "등급",
    when(agro_df["카테고리"] != "축산물", None).otherwise(agro_df["등급"])
)

display(
    agro_df.select("등급").distinct().orderBy("등급")
)

In [0]:
agro_df = agro_df.withColumn("비고", agro_df["등급"])
display(agro_df)

In [0]:
from pyspark.sql.functions import when, col

agro_df = agro_df.withColumn(
    "등급",
    when(col("등급").contains("등급"), col("등급")).otherwise(None)
)
display(agro_df)

In [0]:
agro_df = agro_df.replace("등급란", None, subset=["등급"])

In [0]:
display(agro_df)

In [0]:
from pyspark.sql.functions import to_date, col

agro_df2 = agro_df.withColumn(
    "날짜",
    to_date(col("날짜"), "yyyyMMdd")
)

In [0]:
display(agro_df2)

In [0]:
from pyspark.sql.functions import col

agro_df2 = agro_df2.withColumn("가격", col("가격").cast("int")) \
                   .withColumn("단위_수치", col("단위_수치").cast("int"))

display(agro_df2)

In [0]:
# # agro_df 시도 시군구 분류 csv --> 테이블로 변환
# agro_df2 = spark.read.option("header", "true").csv([
#     "/Volumes/silver/agrofood/agrofood_normalized3/result1.csv",
#     "/Volumes/silver/agrofood/agrofood_normalized3/result2.csv"
# ])
# display(agro_df2)

In [0]:
from pyspark.sql import functions as F

# 시도 정규화 (풀네임 → 짧은 이름)
sido_normalize = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구',
    '인천광역시': '인천', '광주광역시': '광주', '대전광역시': '대전',
    '울산광역시': '울산', '세종특별자치시': '세종',
    '경기도': '경기', '강원도': '강원', '강원특별자치도': '강원',
    '충청북도': '충북', '충청남도': '충남',
    '전라북도': '전북', '전북특별자치도': '전북', '전라남도': '전남',
    '경상북도': '경북', '경상남도': '경남',
    '제주특별자치도': '제주',
}

sido_pattern = '|'.join(sido_normalize.keys())
sido_normalize_expr = F.create_map([F.lit(k) for pair in sido_normalize.items() for k in pair])

# 시도 추출: 문자열 앞부분에서 광역자치단체명 추출
sido_raw = F.regexp_extract(F.col('지역'), rf'^({sido_pattern})', 1)

# 시군구 추출: 시도명 뒤에 오는 시/군/구 단위 추출
# '경기도수원시  권선' → '수원시', '서울특별시송파구' → '송파구'
sigungu_raw = F.regexp_extract(F.col('지역'), rf'^(?:{sido_pattern})([가-힣]+(?:시|군|구))', 1)


지역_idx = agro_df2.columns.index('지역')
cols_before = agro_df2.columns[:지역_idx]
cols_after  = agro_df2.columns[지역_idx + 1:]

agro_df3 = agro_df2.select(
    *cols_before,
    F.when(sido_raw != '', sido_normalize_expr[sido_raw])
     .alias('시도'),
    F.when((sigungu_raw != '') & ~sigungu_raw.isin(list(sido_normalize.keys())), sigungu_raw)
     .otherwise(F.lit(None)).alias('시군구'),
    *cols_after,
).drop('지역')

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

sido_level = {
    '강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산',
    '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북', '전국'
}

# 시군구 원본값 → 시도 매핑
sigungu_to_sido = {
    '수원': '경기', '성남': '경기', '의정부': '경기', '안양': '경기',
    '부천': '경기', '광명': '경기', '평택': '경기', '동두천': '경기',
    '안산': '경기', '고양': '경기', '과천': '경기', '구리': '경기',
    '남양주': '경기', '오산': '경기', '시흥': '경기', '군포': '경기',
    '의왕': '경기', '하남': '경기', '용인': '경기', '파주': '경기',
    '이천': '경기', '안성': '경기', '김포': '경기', '화성': '경기',
    '경기광주': '경기', '양주': '경기', '포천': '경기', '여주': '경기',
    '연천': '경기', '가평': '경기', '양평': '경기',

    '춘천': '강원', '원주': '강원', '강릉': '강원', '동해': '강원',
    '태백': '강원', '속초': '강원', '삼척': '강원', '홍천': '강원',
    '횡성': '강원', '영월': '강원', '평창': '강원', '정선': '강원',
    '철원': '강원', '화천': '강원', '양구': '강원', '인제': '강원',
    '강원고성': '강원', '양양': '강원',

    '청주': '충북', '충주': '충북', '제천': '충북', '보은': '충북',
    '옥천': '충북', '영동': '충북', '증평': '충북', '진천': '충북',
    '괴산': '충북', '음성': '충북', '단양': '충북',

    '천안': '충남', '공주': '충남', '보령': '충남', '아산': '충남',
    '서산': '충남', '논산': '충남', '계룡': '충남', '당진': '충남',
    '금산': '충남', '부여': '충남', '서천': '충남', '청양': '충남',
    '홍성': '충남', '예산': '충남', '태안': '충남',

    '전주': '전북', '군산': '전북', '익산': '전북', '정읍': '전북',
    '남원': '전북', '김제': '전북', '완주': '전북', '진안': '전북',
    '무주': '전북', '장수': '전북', '임실': '전북', '순창': '전북',
    '고창': '전북', '부안': '전북',

    '목포': '전남', '여수': '전남', '순천': '전남', '나주': '전남',
    '광양': '전남', '담양': '전남', '곡성': '전남', '구례': '전남',
    '고흥': '전남', '보성': '전남', '화순': '전남', '장흥': '전남',
    '강진': '전남', '해남': '전남', '영암': '전남', '무안': '전남',
    '함평': '전남', '영광': '전남', '장성': '전남', '완도': '전남',
    '진도': '전남', '신안': '전남',

    '포항': '경북', '경주': '경북', '김천': '경북', '안동': '경북',
    '구미': '경북', '영주': '경북', '영천': '경북', '상주': '경북',
    '문경': '경북', '경산': '경북', '의성': '경북', '청송': '경북',
    '영양': '경북', '영덕': '경북', '청도': '경북', '고령': '경북',
    '성주': '경북', '칠곡': '경북', '예천': '경북', '봉화': '경북',
    '울진': '경북', '울릉': '경북',

    '창원': '경남', '진주': '경남', '통영': '경남', '사천': '경남',
    '김해': '경남', '밀양': '경남', '거제': '경남', '양산': '경남',
    '의령': '경남', '함안': '경남', '창녕': '경남', '경남고성': '경남',
    '남해': '경남', '하동': '경남', '산청': '경남', '함양': '경남',
    '거창': '경남', '합천': '경남',

    '제주시': '제주', '서귀포': '제주',

    '종로구': '서울', '중구': '서울', '용산구': '서울', '성동구': '서울',
    '광진구': '서울', '동대문구': '서울', '중랑구': '서울', '성북구': '서울',
    '강북구': '서울', '도봉구': '서울', '노원구': '서울', '은평구': '서울',
    '서대문구': '서울', '마포구': '서울', '양천구': '서울', '강서구': '서울',
    '구로구': '서울', '금천구': '서울', '영등포구': '서울', '동작구': '서울',
    '관악구': '서울', '서초구': '서울', '강남구': '서울', '송파구': '서울',
    '강동구': '서울',

    '영도구': '부산', '부산진구': '부산', '동래구': '부산',
    '해운대구': '부산', '사하구': '부산', '금정구': '부산',
    '연제구': '부산', '수영구': '부산', '사상구': '부산', '기장군': '부산',

    '수성구': '대구', '달서구': '대구', '달성군': '대구',

    '미추홀구': '인천', '연수구': '인천', '남동구': '인천',
    '부평구': '인천', '계양구': '인천', '강화군': '인천', '옹진군': '인천',

    '광산구': '광주',
    '유성구': '대전', '대덕구': '대전',
    '울주군': '울산',
}

# 시군구 원본값 → 표시 레이블 (접미사 포함)
sigungu_label_map = {
    '수원': '수원시', '성남': '성남시', '의정부': '의정부시', '안양': '안양시',
    '부천': '부천시', '광명': '광명시', '평택': '평택시', '동두천': '동두천시',
    '안산': '안산시', '고양': '고양시', '과천': '과천시', '구리': '구리시',
    '남양주': '남양주시', '오산': '오산시', '시흥': '시흥시', '군포': '군포시',
    '의왕': '의왕시', '하남': '하남시', '용인': '용인시', '파주': '파주시',
    '이천': '이천시', '안성': '안성시', '김포': '김포시', '화성': '화성시',
    '경기광주': '광주시', '양주': '양주시', '포천': '포천시', '여주': '여주시',
    '연천': '연천군', '가평': '가평군', '양평': '양평군',

    '춘천': '춘천시', '원주': '원주시', '강릉': '강릉시', '동해': '동해시',
    '태백': '태백시', '속초': '속초시', '삼척': '삼척시', '홍천': '홍천군',
    '횡성': '횡성군', '영월': '영월군', '평창': '평창군', '정선': '정선군',
    '철원': '철원군', '화천': '화천군', '양구': '양구군', '인제': '인제군',
    '강원고성': '고성군', '양양': '양양군',

    '청주': '청주시', '충주': '충주시', '제천': '제천시', '보은': '보은군',
    '옥천': '옥천군', '영동': '영동군', '증평': '증평군', '진천': '진천군',
    '괴산': '괴산군', '음성': '음성군', '단양': '단양군',

    '천안': '천안시', '공주': '공주시', '보령': '보령시', '아산': '아산시',
    '서산': '서산시', '논산': '논산시', '계룡': '계룡시', '당진': '당진시',
    '금산': '금산군', '부여': '부여군', '서천': '서천군', '청양': '청양군',
    '홍성': '홍성군', '예산': '예산군', '태안': '태안군',

    '전주': '전주시', '군산': '군산시', '익산': '익산시', '정읍': '정읍시',
    '남원': '남원시', '김제': '김제시', '완주': '완주군', '진안': '진안군',
    '무주': '무주군', '장수': '장수군', '임실': '임실군', '순창': '순창군',
    '고창': '고창군', '부안': '부안군',

    '목포': '목포시', '여수': '여수시', '순천': '순천시', '나주': '나주시',
    '광양': '광양시', '담양': '담양군', '곡성': '곡성군', '구례': '구례군',
    '고흥': '고흥군', '보성': '보성군', '화순': '화순군', '장흥': '장흥군',
    '강진': '강진군', '해남': '해남군', '영암': '영암군', '무안': '무안군',
    '함평': '함평군', '영광': '영광군', '장성': '장성군', '완도': '완도군',
    '진도': '진도군', '신안': '신안군',

    '포항': '포항시', '경주': '경주시', '김천': '김천시', '안동': '안동시',
    '구미': '구미시', '영주': '영주시', '영천': '영천시', '상주': '상주시',
    '문경': '문경시', '경산': '경산시', '의성': '의성군', '청송': '청송군',
    '영양': '영양군', '영덕': '영덕군', '청도': '청도군', '고령': '고령군',
    '성주': '성주군', '칠곡': '칠곡군', '예천': '예천군', '봉화': '봉화군',
    '울진': '울진군', '울릉': '울릉군',

    '창원': '창원시', '진주': '진주시', '통영': '통영시', '사천': '사천시',
    '김해': '김해시', '밀양': '밀양시', '거제': '거제시', '양산': '양산시',
    '의령': '의령군', '함안': '함안군', '창녕': '창녕군', '경남고성': '고성군',
    '남해': '남해군', '하동': '하동군', '산청': '산청군', '함양': '함양군',
    '거창': '거창군', '합천': '합천군',

    '제주시': '제주시', '서귀포': '서귀포시',

    '종로구': '종로구', '중구': '중구', '용산구': '용산구', '성동구': '성동구',
    '광진구': '광진구', '동대문구': '동대문구', '중랑구': '중랑구', '성북구': '성북구',
    '강북구': '강북구', '도봉구': '도봉구', '노원구': '노원구', '은평구': '은평구',
    '서대문구': '서대문구', '마포구': '마포구', '양천구': '양천구', '강서구': '강서구',
    '구로구': '구로구', '금천구': '금천구', '영등포구': '영등포구', '동작구': '동작구',
    '관악구': '관악구', '서초구': '서초구', '강남구': '강남구', '송파구': '송파구',
    '강동구': '강동구',

    '영도구': '영도구', '부산진구': '부산진구', '동래구': '동래구',
    '해운대구': '해운대구', '사하구': '사하구', '금정구': '금정구',
    '연제구': '연제구', '수영구': '수영구', '사상구': '사상구', '기장군': '기장군',

    '수성구': '수성구', '달서구': '달서구', '달성군': '달성군',

    '미추홀구': '미추홀구', '연수구': '연수구', '남동구': '남동구',
    '부평구': '부평구', '계양구': '계양구', '강화군': '강화군', '옹진군': '옹진군',

    '광산구': '광산구',
    '유성구': '유성구', '대덕구': '대덕구',
    '울주군': '울주군',
}

def get_sido(region):
    if region in sido_level:
        return region
    return sigungu_to_sido.get(region, None)

def get_sigungu(region):
    if region in sido_level:
        return None
    return sigungu_label_map.get(region, None)

sido_udf    = F.udf(get_sido, StringType())
sigungu_udf = F.udf(get_sigungu, StringType())

cols = agro_df2.columns
지역_idx = list(cols).index('지역')
new_cols = list(cols[:지역_idx]) + ['시도', '시군구'] + list(cols[지역_idx+1:])

agro_df2 = (
    agro_df2
    .withColumn('시도',    sido_udf(F.col('지역')))
    .withColumn('시군구', sigungu_udf(F.col('지역')))
    .drop('지역')
    .select(new_cols)
)

In [0]:
display(agro_df2)

In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace, when, col

agro_df2 = agro_df2.withColumn(
    "비고",
    regexp_extract(col("세부속성"), r"\(([^)]+)\)", 1)
).withColumn(
    "세부속성",
    regexp_replace(col("세부속성"), r"\s*\([^)]+\)", "")
)

display(agro_df2)

In [0]:
agro_df2 = agro_df2.withColumn(
    "재료명",
    when(col("재료명").like("깐마늘%"), "깐마늘").otherwise(col("재료명"))
)
display(agro_df2.select("재료명").distinct().orderBy("재료명"))

In [0]:
display(
    agro_df2.filter(col("카테고리") == "축산물")
           .select("재료명", "세부속성", "등급", "비고")
           .distinct()
           .orderBy("재료명", "세부속성", "등급", "비고")
)

In [0]:
agro_df2 = agro_df2.withColumn(
    "재료명",
    when(col("재료명").like("닭"), "닭고기")
    .when(col("재료명").like("돼지"), "돼지고기")
    .when(col("재료명").like("소"), "소고기")
    .when(col("재료명").like("쇠고기"), "소고기")
    .otherwise(col("재료명"))
)
display(agro_df2.select("재료명").distinct().orderBy("재료명"))

In [0]:
from pyspark.sql.functions import col, regexp_extract, regexp_replace, trim, when

agro_df2 = (
    agro_df2
    # 축산물만 비고/세부속성 변환 (필터 없이 조건부 적용)
    .withColumn(
        "비고",
        when(col("카테고리") == "축산물",
             regexp_extract(col("세부속성"), r"(한우|[가-힣]+산)", 0)
        ).otherwise(col("비고"))
    )
    .withColumn(
        "세부속성",
        when(col("카테고리") == "축산물",
             trim(regexp_replace(col("세부속성"), r"(한우|[가-힣]+산)", ""))
        ).otherwise(col("세부속성"))
    )
)

display(agro_df2)

In [0]:
from pyspark.sql.functions import col, when, regexp_replace

agro_df2 = agro_df2.withColumn(
    "비고",
    when((col("카테고리") == "축산물") & col("재료명").like("수입%"), "수입").otherwise(col("비고"))
).withColumn(
    "재료명",
    when((col("카테고리") == "축산물") & col("재료명").like("수입%"), regexp_replace(col("재료명"), "^수입", "")).otherwise(col("재료명"))
)

display(agro_df2)

In [0]:
from pyspark.sql.functions import trim, when, col

agro_df2 = agro_df2.withColumn(
    "재료명",
    when(trim(col("재료명")) == "돼지고기", "돼지고기")
    .when(trim(col("재료명")) == "소고기", "소고기")
    .otherwise(col("재료명"))
).withColumn(
    "세부속성",
    when(col("세부속성") == "흰우유", "우유").otherwise(col("세부속성"))
)

display(
    agro_df2)

In [0]:
from pyspark.sql.functions import col, when, trim, lit, sum as spark_sum

# 모든 문자열 컬럼에 대해 빈 문자열 또는 공백을 null로 변환
string_cols = [f.name for f in agro_df2.schema.fields if f.dataType.simpleString() == 'string']

# null 처리 전 각 컬럼의 빈값/공백 개수 집계
null_counts = []
for c in string_cols:
    null_count = agro_df2.filter((col(c) == "") | (trim(col(c)) == "")).count()
    null_counts.append((c, null_count))

# null 처리
for c in string_cols:
    agro_df2 = agro_df2.withColumn(
        c,
        when((col(c) == "") | (trim(col(c)) == ""), lit(None)).otherwise(col(c))
    )

# null 처리된 값 개수 출력
for col_name, count in null_counts:
    if count > 0:
        print(f"컬럼 '{col_name}'에서 {count}개 값을 null로 처리함")

In [0]:
#display(agro_df2.distinct().count()) # 12,543,363
display(agro_df2.count()) # 12,558,477

In [0]:
agro_df2.dropDuplicates() \
    .write.mode("overwrite") \
    .saveAsTable("silver.agrofood.agrofood_normalized_fin")

## 테이블 csv로 저장(단일 파일로 저장-coalesce) ----------------

In [0]:
agro_df2.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/silver/agrofood/agrofood_normalized_fin")